# Notebook 02: Community Summaries
## Turning Graph Clusters into Natural-Language Reports

We now have a graph that has been partitioned into **15 communities** (C1 level). Each
community is a cluster of closely-related entities — the 2022 finalists, the host-nation
infrastructure, the FIFA governance controversies, and so on.

In this notebook we ask the LLM to read the entities and relationships of each community
and write a structured **JSON report**. These reports are the bridge between the raw graph
and the final query: in Notebook 03 they feed the Map-Reduce step that answers global
sensemaking questions.

**Paper reference:** Edge et al. (2025) — §3.1.5 *Community Summarization*, Appendix E.2.

### What this notebook does

```
wc2022_trimmed_graph.json  +  wc2022_trimmed_communities.json
           │
           ▼  For each C1 community:
           │    1. Collect entity descriptions + intra-community edge descriptions
           │    2. Prioritise by edge weight (§ 3.1.5)
           │    3. LLM → structured JSON report  (Appendix E.2)
           ▼
data/output/wc2022_trimmed_summaries.json
```

## Imports and Setup

In [1]:
import json
import re
import subprocess
from pathlib import Path

import networkx as nx
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from IPython.display import Markdown, display

Path("../data/output").mkdir(parents=True, exist_ok=True)
print("Imports OK")

Imports OK


In [2]:
PDF_STEM = "wc2022_trimmed"

GRAPH_PATH       = Path("../data/output") / f"{PDF_STEM}_graph.json"
COMMUNITIES_PATH = Path("../data/output") / f"{PDF_STEM}_communities.json"
SUMMARIES_PATH   = Path("../data/output") / f"{PDF_STEM}_summaries.json"

print(f"Graph      : {GRAPH_PATH}")
print(f"Communities: {COMMUNITIES_PATH}")
print(f"Output     : {SUMMARIES_PATH}")

Graph      : ../data/output/wc2022_trimmed_graph.json
Communities: ../data/output/wc2022_trimmed_communities.json
Output     : ../data/output/wc2022_trimmed_summaries.json


### LLM Setup

Same portable Ollama configuration as Notebook 00 (env var → WSL2 → localhost). We increase
`num_predict` to **1500** because community reports include a `findings` array with multiple
entries — they are longer than the extraction tuples.

In [3]:
import os
import platform


def ollama_base_url() -> str:
    """Locate the Ollama server so this notebook runs on any machine.

    Priority: OLLAMA_BASE_URL env var  →  WSL2 Windows-host IP  →  localhost.
    Set OLLAMA_BASE_URL yourself if Ollama runs somewhere else (e.g. a remote box).
    """
    if os.environ.get("OLLAMA_BASE_URL"):
        return os.environ["OLLAMA_BASE_URL"]
    if "microsoft" in platform.uname().release.lower():   # WSL2 → Ollama on the Windows host
        host = subprocess.check_output(
            "ip route list default | awk '{print $3}'", shell=True
        ).decode().strip()
        if host:
            return f"http://{host}:11434"
    return "http://localhost:11434"                        # native Linux / macOS / Windows

MODEL    = "qwen2.5:7b"
BASE_URL = ollama_base_url()

llm = ChatOllama(
    model=MODEL,
    base_url=BASE_URL,
    temperature=0.0,
    num_predict=1500,
)

print(f"LLM: {MODEL}  @  {BASE_URL}")

LLM: qwen2.5:7b  @  http://172.19.64.1:11434


/tmp/ipykernel_55025/2152691020.py:24: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


---
## Step 1 — Load Graph and Communities  *(§ 3.1.5)*

In [4]:
with open(GRAPH_PATH, encoding="utf-8") as f:
    G = nx.node_link_graph(json.load(f))

with open(COMMUNITIES_PATH, encoding="utf-8") as f:
    communities = json.load(f)

c1 = communities["C1"]
print(f"Graph      : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"C1 communities: {len(c1)}")
for cid, members in c1.items():
    intra = sum(1 for u, v in G.edges(members) if v in members)
    print(f"  [{cid}] {len(members):>3} nodes, {intra:>3} intra-edges — {members[:3]}")

Graph      : 179 nodes, 240 edges
C1 communities: 15
  [0]  30 nodes,  32 intra-edges — ['QATAR', '2022 FIFA WORLD CUP', 'AL BAYT STADIUM']
  [1]  21 nodes,  22 intra-edges — ['FIFA', 'RUSSIA', 'AL RAYYAN']
  [2]  19 nodes,  25 intra-edges — ['ARGENTINA', 'FRANCE', 'CROATIA']
  [3]  16 nodes,  24 intra-edges — ['BRAZIL', 'LUSAIL STADIUM', 'STADIUM 974']
  [4]  13 nodes,  15 intra-edges — ['AUSTRALIA', 'POLAND', 'DENMARK']
  [5]  13 nodes,  14 intra-edges — ['AHMAD BIN ALI STADIUM', 'IRAN', 'ALIREZA BEIRANVAND']
  [6]  10 nodes,   9 intra-edges — ['GERMANY', 'UEFA', 'INDONESIA']
  [7]   7 nodes,   7 intra-edges — ['WORLD CUP', 'PRO-GOVERNMENT SUPPORTERS', 'ISRAEL']
  [8]   6 nodes,   7 intra-edges — ['ENZO FERNÁNDEZ', 'KYLIAN MBAPPÉ', 'EMILIANOMARTÍNEZ']
  [9]   5 nodes,   4 intra-edges — ['NEW YORK CITY', 'WAHL', 'QATAR INTERNATIONAL STADIUM']
  [10]   3 nodes,   2 intra-edges — ['SRF INVESTIGATIV', 'PROJECT MERCILESS', 'RAJAT KHARE']
  [11]   3 nodes,   2 intra-edges — ['DOMENICO SCAL

---
## Step 2 — Build Community Context  *(§ 3.1.5)*

For each community we assemble two lists:

1. **Entity descriptions** — one line per node: `NAME (TYPE): description`  
   Sorted by degree descending so the most central entities appear first in the prompt.

2. **Relationship descriptions** — one line per intra-community edge: `SRC → TGT: description`  
   Sorted by `degree(src) + degree(tgt)` descending, as the paper specifies (§3.1.5).
   This ensures the most important relationships fill the context window first.

In [5]:
def build_community_context(G: nx.Graph, members: list[str]) -> tuple[str, str]:
    member_set = set(members)

    # Entity descriptions — sorted by degree (descending)
    nodes_sorted = sorted(
        [n for n in members if n in G.nodes],
        key=lambda n: G.degree(n),
        reverse=True,
    )
    node_lines = [
        f"{n} ({G.nodes[n].get('type', 'UNKNOWN')}): {G.nodes[n].get('description', '')}"
        for n in nodes_sorted
    ]

    # Intra-community edge descriptions — sorted by combined degree (descending)
    intra_edges = [
        (u, v, d)
        for u, v, d in G.edges(members, data=True)
        if u in member_set and v in member_set
    ]
    intra_edges.sort(key=lambda e: G.degree(e[0]) + G.degree(e[1]), reverse=True)
    edge_lines = [
        f"{u} → {v}: {d.get('description', '')}  (weight: {d.get('weight', 1)})"
        for u, v, d in intra_edges
    ]

    return "\n".join(node_lines), "\n".join(edge_lines)


# Quick preview for community [2] (Argentina / France / Croatia / Morocco)
sample_nodes, sample_edges = build_community_context(G, c1["2"])
print("=== Nodes (first 3) ===")
print("\n".join(sample_nodes.split("\n")[:3]))
print("\n=== Edges (first 3) ===")
print("\n".join(sample_edges.split("\n")[:3]))

=== Nodes (first 3) ===
FRANCE (NATION): France is a defending champion who reached their second consecutive World Cup semi-final and defeated Morocco 2-0 in the semi-finals | France is a nation that reached the final of the 2022 FIFA World Cup and lost on penalties to Argentina in the final | France is a nation that has won the tournament since 2002. It also hosts the French national team and is part of UEFA. | France is a national football team that lost their opening game but went on to win the 2022 World Cup | France is a nation that qualified for the knockout stage of the World Cup and won against Australia with four goals. | French national football team, achieved a second World Cup victory after 20 years | France is a nation participating in the 2022 FIFA World Cup and is represented by Kylian Mbappé, the captain of their national team
ARGENTINA (NATION): Argentina advanced to the semi-finals by beating Croatia in extra time and penalties, but lost to the Netherlands in the fina

---
## Step 3 — The Community Summary Prompt  *(Appendix E.2)*

The paper's Appendix E.2 specifies a structured JSON output with five fields. We follow it
verbatim. The `rating` (0–10 float) is particularly important for Notebook 03: when multiple
community summaries are retrieved during the Map step, helpfulness scores determine which
ones feed the final Reduce step.

```
---Goal---
Write a comprehensive report of a community, given a list of entities that
belong to the community and their relationships.

---Report Structure---
Return ONLY a JSON object with exactly these fields:
{
  "title":              "<short, specific title naming key entities>",
  "summary":            "<executive summary, 2–4 sentences>",
  "rating":             <float 0–10, overall importance of this community>,
  "rating_explanation": "<one sentence explaining the rating>",
  "findings": [
    {"summary": "<insight title>", "explanation": "<2–3 sentences>"},
    ...  (3–5 findings)
  ]
}

---Community Data---
Entities:
{node_descriptions}

Relationships:
{edge_descriptions}

Output:
```

In [6]:
SUMMARY_PROMPT = """\
---Goal---
Write a comprehensive report of a community, given a list of entities that belong to the
community and their relationships.

---Report Structure---
Return ONLY a JSON object with exactly these fields:
{{
  "title":              "<short, specific title naming key entities>",
  "summary":            "<executive summary, 2-4 sentences>",
  "rating":             <float 0-10, overall importance of this community>,
  "rating_explanation": "<one sentence explaining the rating>",
  "findings": [
    {{"summary": "<insight title>", "explanation": "<2-3 sentences>"}},
    ... (3-5 findings total)
  ]
}}

---Community Data---
Entities:
{node_descriptions}

Relationships:
{edge_descriptions}

Output:"""


def extract_json(text: str) -> dict:
    """Extract the first JSON object from LLM output, handling markdown fences."""
    # Strip markdown code fences if present
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        return json.loads(m.group(1))
    # Find the outermost { ... }
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        return json.loads(m.group(0))
    raise ValueError(f"No JSON object found in LLM output: {text[:300]}")


def summarise_community(G: nx.Graph, members: list[str]) -> dict:
    node_desc, edge_desc = build_community_context(G, members)
    prompt = SUMMARY_PROMPT.format(
        node_descriptions=node_desc,
        edge_descriptions=edge_desc,
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return extract_json(response.content)


print("Prompt and helpers defined.")

Prompt and helpers defined.


---
## Step 4 — Generate Summaries for All C1 Communities

We iterate over all 15 C1 communities. Expect ~1–2 minutes total on `qwen2.5:7b`.

Each result is appended to `summaries` immediately, so a partial run is not lost if the
kernel is interrupted.

In [7]:
summaries = []

for cid, members in c1.items():
    print(f"Community [{cid}]  ({len(members)} nodes) ...", end=" ", flush=True)
    try:
        report = summarise_community(G, members)
        report["community_id"] = cid
        report["member_count"] = len(members)
        summaries.append(report)
        print(f"OK — {report.get('title', '?')[:60]}")
    except Exception as e:
        print(f"FAILED: {e}")
        summaries.append({
            "community_id": cid,
            "member_count": len(members),
            "title": f"Community {cid} (parse error)",
            "summary": "",
            "rating": 0.0,
            "rating_explanation": "",
            "findings": [],
            "_error": str(e),
        })

print(f"\nDone. {sum(1 for s in summaries if '_error' not in s)}/{len(summaries)} succeeded.")

Community [0]  (30 nodes) ... 

OK — Qatar's 2022 FIFA World Cup: A Mix of Triumph and Controvers
Community [1]  (21 nodes) ... 

OK — FIFA World Cup in Qatar: Community Dynamics and Challenges
Community [2]  (19 nodes) ... 

OK — 2022 FIFA World Cup Final: France vs Argentina
Community [3]  (16 nodes) ... 

OK — 2022 FIFA World Cup Qatar - Brazil and Uruguay's Journey
Community [4]  (13 nodes) ... 

...

OK — Project Merciless and Its Impact on the 2022 World Cup Hosti
Community [11]  (3 nodes) ... 

OK — FIFA Corruption Investigation Network
Community [12]  (2 nodes) ... 

OK — Katara Cultural Village Security Concerns
Community [13]  (2 nodes) ... 

OK — Media-Turf Dispute at 2022 FIFA World Cup
Community [14]  (2 nodes) ... 

OK — Qatar's 2022 World Cup Bid Allegations

Done. 15/15 succeeded.


---
## Step 5 — Save `summaries.json`

In [8]:
# Sort by rating descending so the most important communities appear first
summaries.sort(key=lambda s: s.get("rating", 0), reverse=True)

with open(SUMMARIES_PATH, "w", encoding="utf-8") as f:
    json.dump(summaries, f, ensure_ascii=False, indent=2)

print(f"Saved {SUMMARIES_PATH}  ({SUMMARIES_PATH.stat().st_size / 1024:.1f} KB)")
print(f"\nRatings (community_id — rating — title):")
for s in summaries:
    print(f"  [{s['community_id']}] {s.get('rating', 0):>4.1f}  {s.get('title', '?')[:60]}")

Saved ../data/output/wc2022_trimmed_summaries.json  (24.1 KB)

Ratings (community_id — rating — title):
  [2]  8.5  2022 FIFA World Cup Final: France vs Argentina
  [3]  8.5  2022 FIFA World Cup Qatar - Brazil and Uruguay's Journey
  [8]  8.5  2022 FIFA World Cup Community Report
  [4]  7.5  2022 FIFA World Cup Group D and Related Entities
  [5]  7.5  2022 FIFA World Cup Group B Analysis
  [6]  7.5  2022 FIFA World Cup Bidding Process and Related Entities
  [10]  7.5  Project Merciless and Its Impact on the 2022 World Cup Hosti
  [11]  7.5  FIFA Corruption Investigation Network
  [0]  6.5  Qatar's 2022 FIFA World Cup: A Mix of Triumph and Controvers
  [1]  6.5  FIFA World Cup in Qatar: Community Dynamics and Challenges
  [9]  6.5  Qatar World Cup Security and Media Relations
  [12]  6.5  Katara Cultural Village Security Concerns
  [14]  6.5  Qatar's 2022 World Cup Bid Allegations
  [7]  5.5  2026 FIFA World Cup Community Dynamics
  [13]  4.5  Media-Turf Dispute at 2022 FIFA World Cup


---
## Inspect — Read the Discovered Themes

Below we render each community report as formatted Markdown. This is the knowledge that
the Map-Reduce query in Notebook 03 will draw upon.

In [9]:
for s in summaries:
    if "_error" in s:
        continue
    lines = [
        f"### [{s['community_id']}] {s.get('title', 'Untitled')}",
        f"**Rating:** {s.get('rating', '?')}/10 — {s.get('rating_explanation', '')}",
        f"",
        s.get('summary', ''),
        "",
    ]
    for finding in s.get('findings', []):
        lines.append(f"**{finding.get('summary', '')}**")
        lines.append(finding.get('explanation', ''))
        lines.append("")
    lines.append("---")
    display(Markdown("\n".join(lines)))

### [2] 2022 FIFA World Cup Final: France vs Argentina
**Rating:** 8.5/10 — Highly significant due to the historic nature of the match and the performances of key players like Messi and Mbappé.

The 2022 FIFA World Cup featured a thrilling final between France and Argentina, with Lionel Messi winning the Golden Ball as best player. France's Kylian Mbappé scored eight goals and won the Golden Boot.

**France's Dominant Performance**
France reached their second consecutive World Cup semi-final, defeating Morocco in a penalty shootout. Kylian Mbappé was instrumental with eight goals and the Golden Boot.

**Argentina's Triumph**
Argentina won their third FIFA World Cup title after defeating France 4-2 on penalties. Lionel Messi was named Best Player of the Tournament, cementing his legacy.

**Supreme Committee for Delivery and Legacy's Role**
The Supreme Committee for Delivery and Legacy organized the tournament with measures like sobering-up zones to address alcohol-related issues during the event.

**Group Stage Performances**
Teams like England, Croatia, Morocco, and others showcased their skills in the group stage before advancing or being eliminated based on their performances.

**Key Player Contributions**
Players such as Emiliano Martínez (Argentina) and Adrien Rabiot (France) made significant contributions to their teams' success.

---

### [3] 2022 FIFA World Cup Qatar - Brazil and Uruguay's Journey
**Rating:** 8.5/10 — The community is significant due to Brazil's strong performance and the importance of Lusail Stadium as a venue for crucial matches, including the World Cup final.

Brazil advanced to the knockout stage, defeating Uruguay and Portugal. Uruguay also progressed but lost to Portugal in the group stage. Lusail Stadium hosted several key matches including the final match of the tournament.

**Brazil's Knockout Stage Success**
Brazil advanced to the knockout stage after defeating Uruguay in Group F. They also won against Portugal in the quarter-finals.

**Lusail Stadium's Crucial Role**
Lusail Stadium hosted several important matches, including Brazil’s victory over Uruguay and the final match of the tournament on National Day.

**Uruguay's Group Stage Elimination**
Despite defeating Portugal in the group stage, Uruguay was eliminated from the tournament due to a lower goal difference compared to South Korea.

**Group F Dynamics**
Brazil and Portugal were key teams in Group F. Brazil defeated both Uruguay and Serbia, while Portugal narrowly advanced by defeating Cameroon.

**Stadium 974's Temporary Use**
Stadium 974 hosted matches for several teams including Brazil and Uruguay but was left abandoned after the tournament ended in November 2024.

---

### [8] 2022 FIFA World Cup Community Report
**Rating:** 8.5/10 — The community is highly significant due to the outstanding performances by key players and teams in the tournament.

The community of the 2022 FIFA World Cup is highlighted by key players like Kylian Mbappé and Enzo Fernández, who showcased exceptional skills on the French national team. The report also includes insights into the performance of Senegal and the achievements of Emiliano Martínez.

**Kylian Mbappé's Dominance**
Kylian Mbappé, a star player for France, became the first player since 1966 to score a hat-trick in a World Cup final and won the Golden Boot with eight goals.

**Teamwork on the French National Team**
Mbappé's teammates, including Enzo Fernández and Emiliano Martínez, also excelled, with Fernández winning the Young Player Award and Martínez securing the Golden Glove as the best goalkeeper.

**Senegal's Knockout Performance**
Senegal demonstrated strong team spirit by defeating Ecuador to secure second place in Group A and advance to the knockout stage, showcasing their potential on the global stage.

---

### [4] 2022 FIFA World Cup Group D and Related Entities
**Rating:** 7.5/10 — The community is moderately important as it includes key matches and teams from the World Cup, but the absence of Russia adds a layer of complexity.

The 2022 FIFA World Cup saw Australia, Denmark, Tunisia, and Poland competing in Group D, with notable events including Australia's first win since 2010 against Tunisia. The Russian national team was banned due to non-compliance with WADA rules.

**Australia's First Win Since 2010**
Mitchell Duke scored the only goal in Australia’s match against Tunisia, securing their first World Cup win since 2010 and advancing to the knockout round.

**Russia Banned from Competing**
The Russian national team was banned due to non-compliance with WADA rules, impacting the tournament's dynamics and creating a boycott among other nations like Poland, Sweden, and the Czech Republic.

**Group D Dynamics**
Australia defeated Denmark and Tunisia but lost to Poland in Group D, securing their place in the knockout round as runners-up. The group also included France from Group E.

**Boycott of Matches in Russian Territory**
The Czech Republic and Sweden boycotted matches against Russia due to its invasion of Ukraine, affecting the scheduling and logistics of the tournament.

**Key Players and Events**
Notable players like Craig Goodwin (Australia) and Wahbi Khazri (Tunisia) contributed significant moments in their respective matches during the 2022 FIFA World Cup.

---

### [5] 2022 FIFA World Cup Group B Analysis
**Rating:** 7.5/10 — The group showcased significant geopolitical tensions and unexpected results, making it a noteworthy segment of the tournament.

The 2022 FIFA World Cup Group B featured Iran, the United States, Wales, and Saudi Arabia. The group dynamics were marked by intense competition and unexpected outcomes, particularly with Iran's participation in anti-government protests.

**Iran's Protests and Participation**
During the 2022 FIFA World Cup, Iran faced domestic protests against its government. The team’s participation was overshadowed by these events, with fans displaying slogans like 'Woman, Life, Freedom'.

**United States' Group B Success**
The United States secured their place in the round of 16 after defeating Iran and drawing with England. Their performance highlighted the competitive nature of Group B.

**Saudi Arabia's Knockout Stage Entry**
Despite finishing last in Group A, Saudi Arabia advanced to the knockout stage due to a favorable goal difference, showcasing the importance of this metric in determining outcomes.

**Ahmad Bin Ali Stadium Usage**
Ahmad Bin Ali Stadium hosted two matches during the 2022 FIFA World Cup, including Belgium vs. Canada and Croatia vs. Canada. The stadium's post-tournament modifications were also noted.

**Wales' Elimination**
Wales ended up last in Group B without advancing to the knockout stage, marking a disappointing performance for the team despite their initial draw against the United States.

---

...

### [9] Qatar World Cup Security and Media Relations
**Rating:** 6.5/10 — The community is significant due to its involvement in major international events but faces challenges related to security and media freedom.

The community highlights tensions between Qatar's security forces and media organizations during the 2022 FIFA World Cup, particularly with Iran International. Autopsies in New York City and a tragic death are also noted.

**Security Forces' Actions at QI Stadium**
Qatar's security forces reportedly removed fans wearing certain items, possibly influenced by the denial of entry for critical journalists from Iran International. This suggests a potential conflict between security and media freedom.

**Iran International's Coverage Denial**
Journalists from Iran International were denied entry to cover the 2022 FIFA World Cup at Qatar International Stadium due to their critical stance towards the regime, highlighting tensions in international media relations.

**Use of Mahsa Amini's Name as a Symbol**
The name Mahsa Amini has been used by Iran International as a symbol for freedom and human rights, reflecting the channel's critical stance towards the Iranian regime. This underscores the political context surrounding media coverage.

---

### [12] Katara Cultural Village Security Concerns
**Rating:** 6.5/10 — While the venue was important for the tournament, safety concerns impacted its overall effectiveness and reputation.

The Katara Cultural Village hosted significant events during the 2022 FIFA World Cup but faced notable issues with security, as evidenced by a reporter's experience.

**Security Issues**
A TV reporter encountered security problems while live reporting at Katara Cultural Village, indicating potential vulnerabilities.

**Event Significance**
Despite the challenges, Katara Cultural Village played a crucial role in hosting FIFA World Cup matches and cultural events.

**Community Impact**
The security concerns may have affected the community's perception of safety and reliability during major international events.

---

### [14] Qatar's 2022 World Cup Bid Allegations
**Rating:** 6.5/10 — The controversy over alleged bribes significantly impacts the community's integrity and global perception despite a successful bid outcome.

The community surrounding Qatar’s successful 2022 World Cup bid is marred by allegations of bribery involving key figures, notably Hassan Al Thawadi and Almajid.

**Allegations of Bribery**
Hassan Al Thawadi, leader of Qatar’s 2022 World Cup bid, denied allegations made by an unknown individual (Almajid) regarding bribery to African football officials during the 2010 bid process.

**Impact on Community Integrity**
The ongoing controversy has raised questions about the transparency and ethics of Qatar’s World Cup bid, affecting its global reputation and community trust.

**Denial and Denial Response**
Al Thawadi strongly denied the allegations, which have not been substantiated by any official investigation, leading to a contentious situation within the community.

---

### [7] 2026 FIFA World Cup Community Dynamics
**Rating:** 5.5/10 — While the event had some positive aspects like addressing kosher food needs, it was overshadowed by safety concerns and inter-group conflicts.

The 2026 FIFA World Cup in Qatar was marked by heightened tensions and conflicts among various groups, including Israeli citizens, pro-government supporters, Palestinians, Jewish tourists, Iranian protesters, and religious leaders.

**Safety Concerns for Israeli Citizens**
Israeli citizens faced significant safety issues during the World Cup, with advisories to hide their identity due to potential harassment from various groups.

**Inter-Group Conflicts and Tensions**
The event saw conflicts between Palestinians waving flags and chanting slogans against Israel, leading to clashes with Israeli fans. Additionally, pro-government supporters were involved in harassing protesters displaying anti-government slogans.

**Addressing Religious Needs**
Despite some unfulfilled promises, the establishment of a kosher kitchen under Rabbi Chitrik's supervision helped meet the needs of Jewish tourists and other religious Jews during the tournament.

**Complex Dynamics Between Groups**
The interactions between pro-government supporters, Palestinians, and Israeli citizens highlighted the complex political and social dynamics present in the community during the World Cup.

**Mixed Reception of Qatar's Efforts**
While the Qatari organizers made efforts to accommodate Jewish tourists with kosher food, there were still unfulfilled promises that left some participants unsatisfied.

---

### [13] Media-Turf Dispute at 2022 FIFA World Cup
**Rating:** 4.5/10 — The event underscores issues of press freedom and media access in Qatar, which are significant concerns for global journalism.

The incident involving an Irish RTÉ reporter and Qatari police highlights the ongoing challenges faced by international media in Qatar, particularly during major sporting events.

**Tensions Between Media and Authorities**
The Irish reporter's experience reflects broader tensions between international media and local authorities during the FIFA World Cup.

**Implications for Press Freedom**
This incident raises questions about press freedom in Qatar, a topic of increasing scrutiny from human rights organizations.

**Media Access During Major Events**
The disruption faced by the Irish reporter highlights the challenges media face when covering large-scale international events in certain countries.

---

---
## Summary

| Paper section | What we did |
|---|---|
| § 3.1.5 | Collected node + intra-edge descriptions per community |
| § 3.1.5 | Prioritised edges by combined source+target degree |
| App. E.2 | Used paper's exact JSON output format (title, summary, rating, findings) |

The summaries are saved at `data/output/wc2022_trimmed_summaries.json`, sorted by rating.

**Next → Notebook 03:** We use these summaries in a **Map-Reduce** query. For the question
*"What are the main storylines and themes of the 2022 FIFA World Cup?"*, each summary is
scored for helpfulness (Map), then the top-scoring ones are synthesised into a final answer
(Reduce) — § 3.1.6, Appendix E.3–E.4.